In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler


# 1. 데이터 불러오기 & 전처리
iris = load_iris()


# 1. 데이터셋 준비 (NumPy 데이터를 PyTorch Tensor로 변환)
# 입력 데이터 (9개 샘플, 2개 특성)
# X = torch.tensor([
#    [1.0, 2.0], [1.5, 1.8], [0.8, 2.5],  # Class 0
#    [8.0, 8.0], [7.5, 9.0], [8.5, 7.8],  # Class 1
#    [1.0, 8.0], [1.2, 9.0], [0.5, 8.5]   # Class 2
#], dtype=torch.float32)

X = torch.tensor([load_iris], dtype=torch.float32)

# 타겟 정답 (PyTorch CrossEntropyLoss는 원-핫 인코딩이 아닌 클래스 인덱스(0, 1, 2)를 전달받습니다)
# y = torch.tensor([0, 0, 0, 1, 1, 1, 2, 2, 2], dtype=torch.long)

# ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')


# 2. 신경망 모델 정의 (nn.Module 상속)
class MultiClassNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(MultiClassNet, self).__init__()
        # 계층 정의
        self.fc1 = nn.Linear(input_size, hidden_size)  # 입력층 -> 은닉층
        self.sigmoid = nn.Sigmoid()                    # 활성화 함수
        self.fc2 = nn.Linear(hidden_size, output_size) # 은닉층 -> 출력층

    def forward(self, x):
        out = self.fc1(x)
        out = self.sigmoid(out)
        out = self.fc2(out)  # Softmax는 nn.CrossEntropyLoss 내부에서 처리되므로 생략
        return out

# 3. 모델, 손실 함수, 옵티마이저 생성
input_size = 2
hidden_size = 5
output_size = 3
learning_rate = 0.5

# 재현성을 위한 시드 고정
torch.manual_seed(42)

model = MultiClassNet(input_size, hidden_size, output_size)

# 손실 함수: Softmax + Cross-Entropy가 결합된 형태
criterion = nn.CrossEntropyLoss()

# 최적화 알고리즘: 경사하강법(SGD)
optimizer = optim.SGD(model.parameters(), lr=learning_rate)


# 4. 모델 학습 루프 (Training Loop)
print("=== PyTorch 학습 시작 ===")
epochs = 3000

for epoch in range(epochs):
    # ① 순전파 (Forward)
    outputs = model(X)
    loss = criterion(outputs, y)

    # ② 역전파 (Backward) 및 가중치 업데이트
    optimizer.zero_grad()  # 이전 스텝의 기울기(Gradient) 초기화
    loss.backward()        # 자동 미분을 통해 역전파 수행 (autograd)
    optimizer.step()       # 경사하강법으로 가중치 업데이트

    # 출력
    if (epoch + 1) % 500 == 0:
        # 가장 높은 확률 값을 가진 클래스 인덱스 추출
        _, predicted = torch.max(outputs, 1)
        accuracy = (predicted == y).float().mean() * 100
        print(f"Epoch {epoch + 1:4d} | Loss: {loss.item():.4f} | Accuracy: {accuracy.item():.1f}%")


# 5. 테스트 샘플 예측
print("\n=== 테스트 샘플 예측 ===")
model.eval() # 평가 모드 전환
test_sample = torch.tensor([[1.2, 2.1], [8.1, 8.2], [0.9, 8.8]], dtype=torch.float32)

with torch.no_grad(): # 테스트 단계에서는 기울기 계산 불필요
    logits = model(test_sample)
    # 실제 확률 분포를 보고 싶다면 torch.softmax 적용
    probabilities = torch.softmax(logits, dim=1)
    predictions = torch.argmax(probabilities, dim=1)

for i, (prob, pred) in enumerate(zip(probabilities, predictions)):
    prob_list = [round(p, 3) for p in prob.tolist()]
    print(f"샘플 {i+1} 확률 분포: {prob_list} -> 최종 예측 클래스: Class {pred.item()}")

TypeError: must be real number, not function

In [6]:
"""
아이리스(Iris) 데이터 분류 - PyTorch 활성화 함수 & 확률 예측 실습
--------------------------------------------------------------
1. sklearn에서 Iris 데이터를 불러온다.
2. 신경망(은닉층: ReLU, 출력층: Softmax)을 구성한다.
3. 학습 후, 각 클래스에 속할 "확률"을 출력해본다.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ----------------------------
# 1. 데이터 불러오기 & 전처리
# ----------------------------
iris = load_iris()
X = iris.data          # (150, 4) - 꽃받침/꽃잎 길이·너비
y = iris.target        # (150,)   - 0:setosa, 1:versicolor, 2:virginica
class_names = iris.target_names

# 학습/테스트 분리 (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 표준화 (평균 0, 분산 1) - 신경망 학습 안정화를 위해 중요
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# numpy -> torch tensor 변환
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)


# ----------------------------
# 2. 신경망 모델 정의
# ----------------------------
class IrisNet(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=16, num_classes=3):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = F.relu(self.fc1(x))      # 활성화 함수 1: ReLU
        x = F.relu(self.fc2(x))      # 활성화 함수 2: ReLU
        logits = self.fc3(x)         # 마지막 층은 활성화 함수 없이 raw score(logit) 출력
        return logits                # softmax는 손실함수(CrossEntropyLoss) 내부에서 처리


model = IrisNet()
criterion = nn.CrossEntropyLoss()   # 내부적으로 softmax + NLLLoss 계산
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)


# ----------------------------
# 3. 모델 학습
# ----------------------------
EPOCHS = 500
for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()

    outputs = model(X_train)              # logits
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        with torch.no_grad():
            preds = torch.argmax(outputs, dim=1)
            acc = (preds == y_train).float().mean().item()
        print(f"[Epoch {epoch:3d}] Loss: {loss.item():.4f}  Train Acc: {acc*100:.2f}%")


# ----------------------------
# 4. 테스트 & 확률(softmax) 출력
# ----------------------------
model.eval()
with torch.no_grad():
    test_logits = model(X_test)
    test_probs = F.softmax(test_logits, dim=1)   # 활성화 함수 적용: 확률로 변환
    test_preds = torch.argmax(test_probs, dim=1)
    test_acc = (test_preds == y_test).float().mean().item()

print(f"\n=== 테스트 정확도: {test_acc*100:.2f}% ===\n")

print("샘플별 클래스 확률 (softmax 결과) 확인:")
print(f"{'실제 클래스':<15}{'예측 클래스':<15}{'확률(setosa, versicolor, virginica)'}")
for i in range(10):  # 앞 10개 샘플만 출력
    true_label = class_names[y_test[i].item()]
    pred_label = class_names[test_preds[i].item()]
    probs = test_probs[i].numpy()
    prob_str = ", ".join([f"{p:.3f}" for p in probs])
    print(f"{true_label:<15}{pred_label:<15}[{prob_str}]")

[Epoch  20] Loss: 0.3471  Train Acc: 86.67%
[Epoch  40] Loss: 0.1745  Train Acc: 91.67%
[Epoch  60] Loss: 0.0621  Train Acc: 96.67%
[Epoch  80] Loss: 0.0439  Train Acc: 98.33%
[Epoch 100] Loss: 0.0395  Train Acc: 98.33%
[Epoch 120] Loss: 0.0379  Train Acc: 98.33%
[Epoch 140] Loss: 0.0372  Train Acc: 98.33%
[Epoch 160] Loss: 0.0367  Train Acc: 98.33%
[Epoch 180] Loss: 0.0364  Train Acc: 98.33%
[Epoch 200] Loss: 0.0360  Train Acc: 98.33%
[Epoch 220] Loss: 0.0356  Train Acc: 98.33%
[Epoch 240] Loss: 0.0352  Train Acc: 98.33%
[Epoch 260] Loss: 0.0346  Train Acc: 98.33%
[Epoch 280] Loss: 0.0339  Train Acc: 98.33%
[Epoch 300] Loss: 0.0330  Train Acc: 98.33%
[Epoch 320] Loss: 0.0319  Train Acc: 98.33%
[Epoch 340] Loss: 0.0305  Train Acc: 98.33%
[Epoch 360] Loss: 0.0287  Train Acc: 98.33%
[Epoch 380] Loss: 0.0252  Train Acc: 98.33%
[Epoch 400] Loss: 0.0211  Train Acc: 98.33%
[Epoch 420] Loss: 0.0168  Train Acc: 99.17%
[Epoch 440] Loss: 0.0128  Train Acc: 100.00%
[Epoch 460] Loss: 0.0096  Train